# Transfer Learning (Aprendizado por Transferência) para Diagnóstico de Falhas

**Projeto:** Diagnóstico de Falhas em Rolamentos via CNN + Transformada Wavelet Contínua  
**Dataset:** Escalogramas CWT (CWRU - 4 classes)  
**Objetivo:** Implementar e comparar modelos consolidados de Transfer Learning pré-treinados no ImageNet (**ResNet-18**, **Inception-v3** e **EfficientNet-B0**) com a rede personalizada `BearingCNN` (desenvolvida no Notebook 2), conforme abordado no Referencial Teórico da Monografia (Seção 2.4 e Tabela 4).

---

## 1. Importação de Bibliotecas e Módulos do Projeto

In [ ]:
import sys
from pathlib import Path

# Adicionar a raiz do projeto ao sys.path
BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix

from src.config import PROCESSED_DATA_DIR, BATCH_SIZE, RANDOM_SEED

## 2. Configuração de Hardware e DataLoaders Adaptativos

Modelos como ResNet-18 e EfficientNet-B0 aceitam imagens de entrada $224 \times 224$, enquanto o Inception-v3 exige imagens $299 \times 299$. Criamos uma função utilitária para gerar os DataLoaders no tamanho correto.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo selecionado: {device}')
if device.type == 'cuda':
    print(f'GPU Ativa: {torch.cuda.get_device_name(0)}')

def get_transfer_dataloaders(img_size: int = 224, batch_size: int = BATCH_SIZE):
    """Retorna DataLoaders com o tamanho de imagem especifico do modelo (224 ou 299)."""
    transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_ds = ImageFolder(root=str(PROCESSED_DATA_DIR / 'train'), transform=transform)
    val_ds   = ImageFolder(root=str(PROCESSED_DATA_DIR / 'val'), transform=transform)
    test_ds  = ImageFolder(root=str(PROCESSED_DATA_DIR / 'test'), transform=transform)
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    
    return train_loader, val_loader, test_loader, train_ds.classes

## 3. Função Construtora de Modelos de Transfer Learning

Esta função carrega os pesos pré-treinados do ImageNet (`DEFAULT`), substitui a última camada totalmente conectada (`fc` ou `classifier`) para 4 classes e oferece a opção de congelar as camadas convolucionais (`freeze_backbone=True`).

In [ ]:
def build_transfer_model(model_name: str, num_classes: int = 4, freeze_backbone: bool = True):
    """
    Constrói um modelo de Transfer Learning adaptado para 4 classes.
    
    Modelos Suportados: 'resnet18', 'inception_v3', 'efficientnet_b0'
    """
    model_name = model_name.lower()
    
    if model_name == 'resnet18':
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
        in_features = model.fc.in_features
        model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(in_features, num_classes))
        img_size = 224
        
    elif model_name == 'inception_v3':
        model = models.inception_v3(weights=models.Inception_V3_Weights.DEFAULT, aux_logits=False)
        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
        in_features = model.fc.in_features
        model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(in_features, num_classes))
        img_size = 299  # Inception exige 299x299
        
    elif model_name == 'efficientnet_b0':
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(in_features, num_classes))
        img_size = 224
    else:
        raise ValueError(f'Modelo {model_name} não suportado.')
        
    return model, img_size

## 4. Função Genérica de Treinamento e Avaliação

In [ ]:
def train_and_evaluate_model(model_name: str, num_epochs: int = 5, lr: float = 0.001, freeze: bool = True):
    """Treina e avalia um modelo especifico de Transfer Learning."""
    model, img_size = build_transfer_model(model_name, num_classes=4, freeze_backbone=freeze)
    model = model.to(device)
    
    train_loader, val_loader, test_loader, classes = get_transfer_dataloaders(img_size=img_size)
    
    criterion = nn.CrossEntropyLoss()
    # Treinar apenas parametros que exigem gradiente
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    
    print(f'\n==========================================================')
    print(f' TREINANDO: {model_name.upper()} (Freeze={freeze})')
    print(f'==========================================================')
    
    best_val_acc = 0.0
    
    for epoch in range(1, num_epochs + 1):
        # --- Treino ---
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        for imgs, labels in tqdm(train_loader, desc=f'Época {epoch:02d}/{num_epochs:02d} [Treino]', leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * imgs.size(0)
            _, preds = torch.max(outputs, 1)
            train_total += labels.size(0)
            train_correct += (preds == labels).sum().item()
            
        train_acc = (train_correct / train_total) * 100.0
        
        # --- Validação ---
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * imgs.size(0)
                _, preds = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (preds == labels).sum().item()
                
        val_acc = (val_correct / val_total) * 100.0
        best_val_acc = max(best_val_acc, val_acc)
        
        print(f'Época [{epoch:02d}/{num_epochs:02d}] | Treino Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%')
        
    # --- Avaliação no Teste ---
    model.eval()
    test_preds, test_labels = [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            _, preds = torch.max(outputs, 1)
            test_preds.extend(preds.cpu().numpy())
            test_labels.extend(labels.numpy())
            
    test_acc = (np.array(test_preds) == np.array(test_labels)).mean() * 100.0
    print(f'\n🏆 Acurácia no Teste ({model_name}): {test_acc:.2f}%')
    
    return {
        'model_name': model_name,
        'best_val_acc': best_val_acc,
        'test_acc': test_acc,
        'test_preds': np.array(test_preds),
        'test_labels': np.array(test_labels),
        'classes': classes
    }

## 5. Execução dos Experimentos de Transfer Learning

Rodamos o treinamento para **ResNet-18**, **Inception-v3** e **EfficientNet-B0** para montar a tabela comparativa.

In [ ]:
# Lista de modelos a testar
modelos_para_testar = ['resnet18', 'inception_v3', 'efficientnet_b0']
resultados = {}

for nome_modelo in modelos_para_testar:
    res = train_and_evaluate_model(nome_modelo, num_epochs=5, lr=0.001, freeze=True)
    resultados[nome_modelo] = res

## 6. Tabela Comparativa de Desempenho (Monografia - Tabela 4/7)

Comparamos a acurácia de teste dos modelos de Transfer Learning com a nossa rede `BearingCNN` personalizada (99.89%).

In [ ]:
print('==========================================================')
print(' TABELA COMPARATIVA DE MODELOS (TCC) ')
print('==========================================================')
print(f'{"Modelo":<20s} | {"Acurácia Val (%)":<18s} | {"Acurácia Teste (%)":<18s}')
print('-' * 62)
print(f'{"BearingCNN (Própria)":<20s} | {"99.94%":<18s} | {"99.89%":<18s}')

for nome_modelo, res in resultados.items():
    print(f'{res["model_name"].upper():<20s} | {res["best_val_acc"]:<17.2f}% | {res["test_acc"]:<17.2f}%')
print('==========================================================')